# Amazon India Sales Analysis
**End-to-end data analysis using the supplied original workbook**

This notebook reads the source workbook, uses the `Sheet1` transaction-level data, validates data quality, standardizes text/date fields, and explores sales, profit, order status, product category, payment method, fulfilment, geography, and monthly trends.

> The other workbook sheets are summary/pivot-style sheets. The main analysis is based on the 10,000-row `Sheet1` transaction table.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../Data/Amazon Sales Data India.xlsx")
OUTPUT_DIR = Path("../Analysis/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

xls = pd.ExcelFile(DATA_PATH)
print("Workbook sheets:", xls.sheet_names)
raw = pd.read_excel(DATA_PATH, sheet_name="Sheet1")
raw.head()

## 1. Dataset overview

In [ ]:
print("Shape:", raw.shape)
display(raw.dtypes.to_frame("dtype"))
display(raw.isna().sum().to_frame("missing_values"))
print("Exact duplicate rows:", raw.duplicated().sum())
print("Unique order IDs:", raw["Order_ID"].nunique())

## 2. Data preparation
No rows are removed in this preparation. Text fields are trimmed and `Order_Date` is parsed as a date. Missing values are reported; this source table has no missing values in the inspected columns.

In [ ]:
df = raw.copy()
text_cols = ["Order_ID", "Category", "Product", "Payment_Method", "Fulfillment", "Order_Status", "Ship_State"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
numeric_cols = ["Quantity", "Unit_Price_INR", "Discount_Pct", "Total_Sales_INR", "Profit_INR"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("Missing values after standardization:")
display(df.isna().sum().to_frame("missing"))
print("Duplicate rows:", df.duplicated().sum())

## 3. Key performance indicators

In [ ]:
kpis = {
    "Rows / orders": len(df),
    "Unique order IDs": df["Order_ID"].nunique(),
    "Total sales (INR)": df["Total_Sales_INR"].sum(),
    "Total profit (INR)": df["Profit_INR"].sum(),
    "Units sold": df["Quantity"].sum(),
    "Average order value (INR)": df["Total_Sales_INR"].sum() / df["Order_ID"].nunique(),
    "Profit margin (%)": df["Profit_INR"].sum() / df["Total_Sales_INR"].sum() * 100,
    "Average discount (%)": df["Discount_Pct"].mean() * 100,
}
pd.Series(kpis, name="Value").to_frame()

## 4. Category performance

In [ ]:
category = df.groupby("Category").agg(
    Orders=("Order_ID", "nunique"),
    Units=("Quantity", "sum"),
    Sales_INR=("Total_Sales_INR", "sum"),
    Profit_INR=("Profit_INR", "sum")
).sort_values("Sales_INR", ascending=False)
category["Sales_Share_pct"] = category["Sales_INR"] / df["Total_Sales_INR"].sum() * 100
display(category)
category["Sales_INR"].sort_values().plot(kind="barh", title="Sales by category (INR)", figsize=(9,5))
plt.xlabel("Sales (INR)"); plt.tight_layout(); plt.show()

## 5. Order status, fulfilment and payment method

In [ ]:
status = df["Order_Status"].value_counts().to_frame("Orders")
status["Share_pct"] = status["Orders"] / len(df) * 100
display(status)
display(df.groupby("Fulfillment").agg(Orders=("Order_ID","nunique"), Sales_INR=("Total_Sales_INR","sum"), Profit_INR=("Profit_INR","sum")).sort_values("Sales_INR", ascending=False))
display(df.groupby("Payment_Method").agg(Orders=("Order_ID","nunique"), Sales_INR=("Total_Sales_INR","sum")).sort_values("Sales_INR", ascending=False))

## 6. State and product performance

In [ ]:
state = df.groupby("Ship_State").agg(Orders=("Order_ID","nunique"), Sales_INR=("Total_Sales_INR","sum"), Profit_INR=("Profit_INR","sum")).sort_values("Sales_INR", ascending=False)
top_products = df.groupby("Product").agg(Orders=("Order_ID","nunique"), Units=("Quantity","sum"), Sales_INR=("Total_Sales_INR","sum"), Profit_INR=("Profit_INR","sum")).sort_values("Sales_INR", ascending=False)
display(state)
display(top_products.head(10))

## 7. Monthly trend

In [ ]:
monthly = df.set_index("Order_Date").resample("MS").agg(
    Orders=("Order_ID","nunique"),
    Sales_INR=("Total_Sales_INR","sum"),
    Profit_INR=("Profit_INR","sum")
)
display(monthly.head())
monthly["Sales_INR"].plot(figsize=(11,4), title="Monthly sales (INR)")
plt.ylabel("Sales (INR)"); plt.tight_layout(); plt.show()

## 8. Export cleaned data and summary tables

In [ ]:
df.to_csv(OUTPUT_DIR / "Amazon_Sales_Cleaned.csv", index=False)
category.to_csv(OUTPUT_DIR / "sales_by_category.csv")
status.to_csv(OUTPUT_DIR / "order_status_summary.csv")
state.to_csv(OUTPUT_DIR / "sales_by_state.csv")
top_products.to_csv(OUTPUT_DIR / "product_performance.csv")
monthly.to_csv(OUTPUT_DIR / "monthly_sales_trend.csv")
print("Saved outputs to", OUTPUT_DIR.resolve())